# PDF Parser

In [23]:
import pymupdf4llm
import re
import httpx
import nest_asyncio
nest_asyncio.apply()

import asyncio


In [24]:
md = pymupdf4llm.to_markdown("../data/SAMPLE_CV.pdf",page_chunks=True)

In [25]:
md[0]

{'metadata': {'format': 'PDF 1.7',
  'title': '',
  'author': 'fiqih.fathor.rachim@gmail.com',
  'subject': '',
  'keywords': '',
  'creator': 'Microsoft® Word 2024',
  'producer': 'Microsoft® Word 2024',
  'creationDate': "D:20260303213335+07'00'",
  'modDate': "D:20260303213335+07'00'",
  'trapped': '',
  'encryption': None,
  'file_path': '../data/SAMPLE_CV.pdf',
  'page_count': 2,
  'page': 1},
 'toc_items': [],
 'tables': [],
 'images': [],
 'graphics': [],
 'text': '# **FIQIH FATHOR RACHIM**\n\nAI Engineer with 3 years of experience at Bank Mega, specializing in building advanced AI systems, robust APIs, and scalable Python\nautomation to drive innovation in the financial sector. Expertly developed solutions in optical character recognition, natural language\nprocessing, and intelligent document processing. Proven track record in seamless AI system integration, robust API development, and\nefficient workflow automation leveraging modern frameworks like PyTorch, FastAPI, and n8n.\

# Chunk

In [26]:
def _split_markdown_sections(text: str) -> list[str]:
    """Split markdown text by headings (# ## ###)."""
    pattern = r"(?=^#{1,3}\s)"
    sections = re.split(pattern, text, flags=re.MULTILINE)
    return [s.strip() for s in sections if s.strip()]

In [27]:
def _split_by_sentences(text: str, chunk_size: int, overlap: int) -> list[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current = []
    current_len = 0

    for sentence in sentences:
        words = sentence.split()
        if current_len + len(words) > chunk_size and current:
            chunks.append(" ".join(current))
            overlap_sentences = []
            overlap_len = 0
            for s in reversed(current):
                s_len = len(s.split())
                if overlap_len + s_len <= overlap:
                    overlap_sentences.insert(0, s)
                    overlap_len += s_len
                else:
                    break
            current = overlap_sentences
            current_len = overlap_len

        current.extend(words)
        current_len += len(words)

    if current:
        chunks.append(" ".join(current))

    return chunks


In [28]:
def chunk_pages(
    pages: list[dict],
    chunk_size: int = 512,
    overlap: int = 64,
) -> list[dict]:
    chunks = []
    global_idx = 0

    for page in pages:
        sections = _split_markdown_sections(page["text"])

        if not sections:
            sections = [page["text"]]

        for section in sections:
            words = section.split()

            if len(words) <= chunk_size:
                chunks.append({
                    "text" : section,
                    "page_number":page["metadata"]["page"],
                    "filename":page["metadata"]["file_path"],
                    "chunk_index":global_idx,
                })
                global_idx += 1
            else:
                sub_chunks = _split_by_sentences(section, chunk_size, overlap)
                for sub in sub_chunks:
                    chunks.append({
                        "text" : sub,
                        "page_number":page["metadata"]["page"],
                        "filename":page["metadata"]["file_path"],
                        "chunk_index":global_idx,
                    })
                    global_idx += 1

    return chunks

In [29]:
chunk = chunk_pages(md)

In [30]:
md

[{'metadata': {'format': 'PDF 1.7',
   'title': '',
   'author': 'fiqih.fathor.rachim@gmail.com',
   'subject': '',
   'keywords': '',
   'creator': 'Microsoft® Word 2024',
   'producer': 'Microsoft® Word 2024',
   'creationDate': "D:20260303213335+07'00'",
   'modDate': "D:20260303213335+07'00'",
   'trapped': '',
   'encryption': None,
   'file_path': '../data/SAMPLE_CV.pdf',
   'page_count': 2,
   'page': 1},
  'toc_items': [],
  'tables': [],
  'images': [],
  'graphics': [],
  'text': '# **FIQIH FATHOR RACHIM**\n\nAI Engineer with 3 years of experience at Bank Mega, specializing in building advanced AI systems, robust APIs, and scalable Python\nautomation to drive innovation in the financial sector. Expertly developed solutions in optical character recognition, natural language\nprocessing, and intelligent document processing. Proven track record in seamless AI system integration, robust API development, and\nefficient workflow automation leveraging modern frameworks like PyTorch,

In [31]:
chunk[1]

{'text': '**BI-OJK Hackathon 2025: BI-Rise, Semifinalist (Top 10 of 2,336 Applicants)**\n\n\n\n\n   - Engineered a self-learning fraud detection system that adapted to new threats by automatically generating rules\nfrom missed detection patterns.\n\n   - Implemented intelligent investigation chatbot that analyzes user behavior across transaction history, blacklist data,\nuser profiles, and graph network relationships.\n\n   - Developed explainable AI rule engine using Domain Specific Language with human-readable descriptions, rule\ncategorization, and audit trails for regulatory compliance\n\n   - Built automated PDF report generation system enabling conversational fraud investigation with comprehensive\nmulti-dimensional analysis and exportable documentation\n\n   - Architected real-time adaptive learning pipeline bridging B-Score risk assessment and AI Rule Agent to\ncontinuously improve fraud detection accuracy through automated rule discovery\n\n**Financial Document Classification*

# Embedder

In [32]:
async def embed_texts(texts: list[str]) -> list[list[float]]:
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.post(
            "http://localhost:8001/embed",
            json={"inputs": texts},
        )
        response.raise_for_status()

    vectors = response.json()
    return vectors

In [33]:
async def embed_query(query: str) -> list[float]:
    vectors = await embed_texts([query])
    return vectors[0]

In [34]:
test_text =  ["aku sedang mengerjakan project rag","dalam waktu 5 hari"]
vector_1 = asyncio.run(embed_texts(test_text))
print(len(vector_1))
print(len(vector_1[0]))

2
1024


In [41]:
test_query =  "Sedang apakah kamu?"
vector_2 = asyncio.run(embed_query(test_query))
print(len(vector_2))
print(vector_2)

1024
[0.010042502, -0.065814696, -0.004226115, -0.047649708, 0.0033391528, -0.035662152, -0.011520077, 0.056364898, -0.026078783, -0.053459834, -0.018331949, -0.014633836, 0.038867738, -0.0046289004, -0.030803684, 0.01749716, 0.029584892, 0.02479321, 0.06755106, -0.06985507, -0.004165593, -0.022923283, -0.0057976036, 0.058234822, -0.0445443, 0.04107158, -0.014959404, 0.12702137, -0.011261293, 0.010217807, 0.07072325, 0.0143834, -0.038133122, 0.00622752, 0.02789862, -0.0056181243, 0.03960235, -0.03140473, -0.0048292493, 0.0069454378, -0.0043909857, -0.012054341, 0.06310999, -0.015318363, 0.007133265, -0.020986574, 0.021570927, -0.010585114, -0.01848221, 0.024008507, -0.029518109, -0.021671101, -0.017346898, 0.0013753135, -0.01758064, -0.06838585, 0.008385447, -0.0051089036, 0.014625489, -0.020368831, -0.06571452, 0.053326268, -0.038166516, 0.015911063, 0.0021767102, 0.047983624, -0.0137072215, -0.035027713, -0.018031424, 0.0024501032, 0.017831076, 0.012672084, 0.0063903034, 0.029017236,